# 🧪 W7-D2 概念实验：Agent 怎么记住、找回、又忘掉？

> 配套阅读：`第7周-Day2-记忆系统与语义搜索.md`（三层结构讲解、MEMORY.md 规范在那边）
> 本 notebook 回答：**记忆不是『把一切存下来』，而是分层存放 + 按语义召回 + 主动遗忘**。用 4 个纯模拟实验拆开验证。
>
> 实验环境：纯 numpy/matplotlib/标准库，无真实 embedding 模型（用词表向量代替原理验证）。


## 实验 1：三层记忆模拟 —— 同一事件流，工作/情景/语义各留住了什么？

工作记忆（上下文窗口）容量最小；情景记忆随时间衰减；语义记忆只沉淀**重复出现**的稳定事实。
实现一个三层记忆系统，灌入 30 条事件流，看每层最后剩下什么。


In [ ]:
from collections import deque, Counter

class WorkingMemory:          # 短期：上下文窗口，FIFO 定容
    def __init__(self, cap=5):
        self.buf = deque(maxlen=cap)
    def add(self, item): self.buf.append(item)

class EpisodicMemory:         # 中期：会话历史，检索权重随时间衰减
    def __init__(self, tau=10.0):
        self.events = []; self.tau = tau
    def add(self, item, t): self.events.append((item, t))
    def recall(self, now, topk=3):
        scored = [(item, pow(2.718281828, -(now - t) / self.tau)) for item, t in self.events]
        return sorted(scored, key=lambda x: -x[1])[:topk]

class SemanticMemory:         # 长期：重复 >= 2 次才晋升为稳定事实
    def __init__(self, promote_at=2):
        self.promote_at = promote_at
        self.counter = Counter(); self.facts = set()
    def add(self, item):
        self.counter[item] += 1
        if self.counter[item] >= self.promote_at and item not in self.facts:
            self.facts.add(item)

wm, em, sm = WorkingMemory(), EpisodicMemory(), SemanticMemory()
stream = [
    (1, "用户问好"), (2, "偏好:中文回答"), (3, "问W3的RAG"), (4, "偏好:喜欢表格"),
    (5, "闲聊天气"), (6, "偏好:中文回答"), (8, "问退款政策"), (9, "事实:Jason在学W7"),
    (11, "偏好:喜欢表格"), (13, "事实:Jason在学W7"), (15, "闲聊周末"), (17, "问记忆系统"),
    (19, "偏好:先结论后展开"), (21, "偏好:中文回答"), (23, "问DAG编排"), (25, "偏好:先结论后展开"),
    (28, "事实:偏好Mac快捷键"), (29, "闲聊新闻"), (30, "问评估体系"),
]
for t, item in stream:
    wm.add(item); em.add(item, t)
    if item.startswith(("偏好", "事实")): sm.add(item)

print(f"工作记忆（容量5）只剩最近5条：{list(wm.buf)}")
print(f"\n情景记忆按 recency 检索 top3：{[(i, round(w,3)) for i, w in em.recall(now=30)]}")
print(f"\n语义记忆沉淀了 {len(sm.facts)} 条稳定事实：{sorted(sm.facts)}")
print("\n观察：一次性闲聊全部蒸发；重复出现的偏好被晋升；单次出现的事实(如Mac快捷键)未过晋升线。")

## 实验 2：遗忘曲线 —— 不复习 / 固定复习 / 间隔递增复习，30 天后谁还记得？

艾宾浩斯遗忘曲线近似指数衰减。复习能把强度重置为 1，而**间隔递增**（1→3→7→15 天）还会逐次拉长记忆的时间常数（巩固效应）。
模拟三种策略 30 天的记忆强度轨迹。


In [ ]:
import numpy as np
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

days = np.linspace(0, 30, 600)

def strength(trace, tau0=2.0, boost=1.9):
    """trace: 复习日列表。每次复习强度重置为1，tau 增大（巩固）"""
    out = np.zeros_like(days); tau = tau0; last = 0.0
    checkpoints = sorted(trace + [999])
    for cp in checkpoints:
        mask = (days >= last) & (days < cp)
        out[mask] = np.exp(-(days[mask] - last) / tau)
        if cp <= 30:
            tau *= boost; last = cp; out[days == cp] = 1.0
    mask = days >= last
    out[mask] = np.exp(-(days[mask] - last) / tau)
    return out

strategies = {
    "不复习": strength([]),
    "固定每5天复习": strength([5, 10, 15, 20, 25], boost=1.0),
    "间隔递增(1,3,7,15)": strength([1, 3, 7, 15]),
}
plt.figure(figsize=(10, 4.5))
for label, s in strategies.items():
    plt.plot(days, s, label=label, lw=2)
plt.axhline(0.3, color="gray", ls="--", lw=1)
plt.text(30.2, 0.3, "遗忘阈值", fontsize=9, va="center")
plt.xlabel("天数"); plt.ylabel("记忆强度"); plt.title("遗忘曲线与三种复习策略")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print("第30天强度：", {k: round(float(v[-1]), 3) for k, v in strategies.items()})
print("→ 记忆系统的复习/重写节奏应该模仿间隔递增，而不是等强度跌没了再抢救。")

## 实验 3：语义搜索 vs 关键词匹配 —— 为什么「退款流程」能召回写着「退换货政策」的记忆？

语义搜索的底层：把文本映射到向量，用 cosine similarity 比较**意思**而不是字面。
用手工词表构造分组向量（同义词共享维度），验证语义召回能跨过字面差异，而精确关键词匹配会漏。


In [ ]:
import numpy as np

GROUPS = {
    "售后": ["退货", "换货", "退款", "退换货", "售后"],
    "物流": ["物流", "快递", "发货", "签收"],
    "偏好": ["偏好", "风格", "格式", "中文"],
    "账号": ["账号", "密码", "登录"],
}
DIM = list(GROUPS)

def embed(text):
    """分组词表向量：命中某组任意词 → 该维 +1（模拟同义共现的 embedding）"""
    v = np.zeros(len(DIM))
    for i, g in enumerate(DIM):
        v[i] = sum(text.count(w) for w in GROUPS[g])
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

memories = [
    "退换货政策：需提供订单号，7天内可办",
    "物流时效：默认顺丰，3日达",
    "用户偏好：中文回答、先结论后展开",
    "账号安全：连续错误5次将锁定",
    "用户偏好：喜欢表格和清单",
    "签收后异常件走售后通道",
]

def cosine(a, b): return float(np.dot(a, b))

queries = ["退款流程怎么走", "我要查快递", "他喜欢什么样的回答格式"]
for q in queries:
    qv = embed(q)
    ranked = sorted(memories, key=lambda m: -cosine(qv, embed(m)))
    keywords = [w for words in GROUPS.values() for w in words if w in q]  # 词级匹配
    kw_hits = [m for m in memories if any(w in m for w in keywords)]
    print(f"查询: {q}")
    print(f"  关键词精确匹配(词={keywords}): {kw_hits if kw_hits else '（无命中）'}")
    print(f"  语义 top-2      : {ranked[:2]}")
    print(f"  top-1 cosine    : {cosine(qv, embed(ranked[0])):.3f}\n")

print("→ 「退款流程」字面不在「退换货政策」里，关键词漏了；语义向量因同组词共现而召回成功。")

## 实验 4：写入决策 —— 该记什么、该忘什么？频率×新鲜度打分 + 阈值淘汰

MEMORY.md 不能变成垃圾场：稳定偏好和重复事实才值得沉淀，一次性噪声应被过滤。
构造 100 条事件流（60 条一次性噪声 + 重复偏好 + 重复事实），比较「全记」vs「过滤后」的噪声率。


In [ ]:
import random
from collections import Counter

random.seed(11)

PREDS = ["偏好:中文回答", "偏好:喜欢表格", "偏好:先结论", "偏好:Mac快捷键"]
FACTS = ["事实:Jason学完W1-W6", "事实:W7主线是数字员工", "事实:用Python做实验"]
events = []
for _ in range(60):
    events.append("噪声:" + "".join(random.choices("abcdefgh", k=6)))   # 一次性细节
for _ in range(25):
    events.append(random.choice(PREDS))
for _ in range(15):
    events.append(random.choice(FACTS))

freq = Counter(events)
n = len(events)
scored = []
for pos, e in enumerate(events):
    recency = (pos + 1) / n
    score = 0.7 * min(freq[e] / 4, 1.0) + 0.3 * recency       # 频次为主，新鲜度辅助
    scored.append((e, score))

# 规则：重复出现(freq>=2)是写入长期记忆的必要条件，分数只做排序
kept = [e for e, s in scored if freq[e] >= 2 and s >= 0.35]
dedup = list(dict.fromkeys(kept))

def noise_ratio(items):
    if not items: return 0.0
    return sum(i.startswith("噪声") for i in items) / len(items)

print(f"全记（不去重不去噪）：{len(events)} 条，噪声率 {noise_ratio(events):.0%}")
print(f"过滤+去重后          ：{len(dedup)} 条，噪声率 {noise_ratio(dedup):.0%}（重复必要条件挡掉全部一次性噪声）")
print(f"\n拟写入 MEMORY.md 的候选：")
for item in dedup:
    print(f"  - {item}")
print("\n→ 打分规则让『重复出现』成为沉淀的主要信号，单次噪声自然被阈值挡在门外。")

## 结论

| 机制 | 实验现象 |
|---|---|
| 三层记忆 | 闲聊蒸发、偏好晋升、情景按 recency 衰减（实验1） |
| 遗忘与复习 | 间隔递增 >> 固定复习 >> 不复习（实验2） |
| 语义召回 | 同组词跨字面差异命中，关键词匹配漏检（实验3） |
| 写入决策 | 频率×新鲜度打分把噪声率从 ~60% 压到 0（实验4） |

**记忆系统 = 分层存放 + 语义召回 + 主动遗忘，三者缺一不可。**
→ 深入阅读：同名 `.md` 的 MEMORY.md 规范、memory/*.md 分类与 OpenClaw vs Hermes 对比。
